In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname

#RS
from torch.utils.data import WeightedRandomSampler



root_path = dirname(os.getcwd()) + "/SEPH_OUTCOME"

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print("CWD:", os.getcwd())
print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = "cpu"

CWD: /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/graphs_repair/


In [2]:
with open("data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [3]:
list(datasets_info.keys())

['BPIC11_f1',
 'sepsis_cases_1',
 'sepsis_cases_2',
 'BPIC15_3_f2',
 'bpic2012_O_DECLINED-COMPLETE',
 'traffic_fines_1',
 'BPIC17_O_Cancelled',
 'hospital_billing_3']

In [4]:
dataset = "sepsis_cases_2" #Select dataset to work on

In [5]:
#if dataset.startswith("BPIC15"):
#    with open("data/dataset_features.json", 'r') as file:
#        dataset_info = json.load(file)["BPIC15_common"]
#else:
#    with open("data/dataset_features.json", 'r') as file:
#        dataset_info = json.load(file)[dataset]


with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]

In [6]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [7]:
tab_all = pd.read_csv(data_dir_processed+dataset+"_processed_all.csv")
tab_all.head()

,Diagnose,DiagnosticArtAstrup,DiagnosticBlood,DiagnosticECG,DiagnosticIC,DiagnosticLacticAcid,DiagnosticLiquor,DiagnosticOther,DiagnosticSputum,DiagnosticUrinaryCulture,DiagnosticUrinarySediment,DiagnosticXthorax,DisfuncOrg,Hypotensie,Hypoxie,InfectionSuspected,Infusion,Oligurie,SIRSCritHeartRate,SIRSCritLeucos,SIRSCritTachypnea,SIRSCritTemperature,SIRSCriteria2OrMore,Age,CaseID,Activity,org:group,CRP,LacticAcid,Leucocytes,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases,Label
0,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,ER Registration,A,0.0,0.0,0.0,1.413955e+09,555,10,2,9,0.000000,0.000000,1,81,regular
1,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,Leucocytes,B,0.0,0.0,9.6,1.413956e+09,567,10,2,9,0.000000,11.316667,2,81,regular
2,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,CRP,B,21.0,0.0,9.6,1.413956e+09,567,10,2,9,0.000000,11.316667,3,81,regular
3,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,LacticAcid,B,21.0,2.2,9.6,1.413956e+09,567,10,2,9,11.316667,11.316667,4,81,regular
4,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,ER Triage,C,21.0,2.2,9.6,1.413956e+09,573,10,2,9,6.616667,17.933333,5,81,regular


In [8]:
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [9]:
tab_test = pd.read_csv(f"data/datasets/processed/{dataset}_processed_test.csv")
minority_lengths = tab_test[tab_test["Label"]=="deviant"].groupby("CaseID").size()
total_minority = len(minority_lengths)
q90 = int(np.ceil(minority_lengths.quantile(0.9)))
MaxPrefix = min(40,q90)
print(f"90th‐percentile threshold: {q90:.2f}  MaxPrefix: {MaxPrefix}")

90th‐percentile threshold: 16.00  MaxPrefix: 16


In [10]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "rb") as f:
    X_train = pickle.load(f)
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "rb") as f:
    X_valid = pickle.load(f)
with open(data_dir_graphs + dataset + "_TEST4_repair.pkl", "rb") as f:
    X_test = pickle.load(f)

X_tests = {}
for L in range(1, MaxPrefix + 1):
    fname = f"{dataset}_TEST{L}_repair.pkl"
    path  = os.path.join(data_dir_graphs, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Expected file not found: {path}")
    with open(path, "rb") as f:
        X_tests[L] = pickle.load(f)
    print(f"Loaded {len(X_tests[L])} graphs for prefix length {L} from {fname}")

Loaded 157 graphs for prefix length 1 from sepsis_cases_2_TEST1_repair.pkl
Loaded 157 graphs for prefix length 2 from sepsis_cases_2_TEST2_repair.pkl
Loaded 157 graphs for prefix length 3 from sepsis_cases_2_TEST3_repair.pkl
Loaded 157 graphs for prefix length 4 from sepsis_cases_2_TEST4_repair.pkl
Loaded 156 graphs for prefix length 5 from sepsis_cases_2_TEST5_repair.pkl
Loaded 155 graphs for prefix length 6 from sepsis_cases_2_TEST6_repair.pkl
Loaded 154 graphs for prefix length 7 from sepsis_cases_2_TEST7_repair.pkl
Loaded 154 graphs for prefix length 8 from sepsis_cases_2_TEST8_repair.pkl
Loaded 143 graphs for prefix length 9 from sepsis_cases_2_TEST9_repair.pkl
Loaded 135 graphs for prefix length 10 from sepsis_cases_2_TEST10_repair.pkl
Loaded 124 graphs for prefix length 11 from sepsis_cases_2_TEST11_repair.pkl
Loaded 113 graphs for prefix length 12 from sepsis_cases_2_TEST12_repair.pkl
Loaded 97 graphs for prefix length 13 from sepsis_cases_2_TEST13_repair.pkl
Loaded 77 graphs f

In [11]:

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected, NormalizeFeatures

transform = ToUndirected()

with torch.no_grad():
        for i in range(len(X_train)):
                X_train[i] = transform(X_train[i])
        for i in range(len(X_valid)):
                X_valid[i] = transform(X_valid[i])
        for L, graphs_L in X_tests.items():
                for i in range(len(graphs_L)):
                        graphs_L[i] = transform(graphs_L[i])
    


In [12]:
edge_types = set()
node_types = set()
for i in range(len(X_train)):
    n, edge_type = X_train[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_valid)):
    n, edge_type = X_valid[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for graphs_L in X_tests.values():
    for i in range(len(graphs_L)):
        n, edge_type = graphs_L[i].metadata()
        for x in n:
            node_types.add(x)
        for x in edge_type:
            edge_types.add(x)



In [13]:
node_types = list(node_types)
edge_types = list(edge_types)

In [14]:
node_types

['open_cases',
 'DiagnosticArtAstrup',
 'DiagnosticLacticAcid',
 'DiagnosticIC',
 'hour',
 'org:group',
 'Age',
 'DiagnosticXthorax',
 'Infusion',
 'DisfuncOrg',
 'LacticAcid',
 'Diagnose',
 'timesincecasestart',
 'time:timestamp',
 'SIRSCritLeucos',
 'weekday',
 'event_nr',
 'DiagnosticECG',
 'timesincelastevent',
 'DiagnosticUrinarySediment',
 'Activity',
 'InfectionSuspected',
 'SIRSCritHeartRate',
 'Hypoxie',
 'DiagnosticBlood',
 'DiagnosticOther',
 'CRP',
 'Oligurie',
 'DiagnosticLiquor',
 'timesincemidnight',
 'DiagnosticSputum',
 'SIRSCritTachypnea',
 'Hypotensie',
 'SIRSCriteria2OrMore',
 'DiagnosticUrinaryCulture',
 'month',
 'SIRSCritTemperature',
 'Leucocytes']

In [15]:
edge_types

[('Activity', 'related_to', 'time:timestamp'),
 ('Hypoxie', 'related_to', 'Hypoxie'),
 ('Activity', 'followed_by', 'Activity'),
 ('SIRSCritHeartRate', 'related_to', 'SIRSCritHeartRate'),
 ('SIRSCriteria2OrMore', 'related_to', 'SIRSCriteria2OrMore'),
 ('Activity', 'related_to', 'DiagnosticECG'),
 ('Age', 'rev_related_to', 'Activity'),
 ('weekday', 'rev_related_to', 'Activity'),
 ('CRP', 'related_to', 'CRP'),
 ('SIRSCritTachypnea', 'rev_related_to', 'Activity'),
 ('weekday', 'related_to', 'weekday'),
 ('SIRSCriteria2OrMore', 'rev_related_to', 'Activity'),
 ('org:group', 'rev_related_to', 'Activity'),
 ('Activity', 'related_to', 'Oligurie'),
 ('Activity', 'related_to', 'SIRSCriteria2OrMore'),
 ('month', 'related_to', 'month'),
 ('Activity', 'related_to', 'LacticAcid'),
 ('Infusion', 'related_to', 'Infusion'),
 ('Activity', 'related_to', 'InfectionSuspected'),
 ('Activity', 'related_to', 'timesincecasestart'),
 ('open_cases', 'rev_related_to', 'Activity'),
 ('DiagnosticXthorax', 'rev_relat

## Hyperopt

In [16]:
print(f"PyTorch: {torch.__version__}")
#print(f"TorchVision: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch: 2.6.0+cu124
CUDA Available: False


In [17]:
from ax.service.managed_loop import optimize

In [18]:
from torch_geometric.nn import (
    HeteroConv,
    global_mean_pool,
    GATv2Conv,
    SAGEConv,
    TransformerConv
)
from torch.nn import (
    ModuleList,
    Module,
    Linear
  )
from typing_extensions import Self

In [19]:
from torch_geometric.nn import HeteroConv, global_mean_pool, SAGEConv
from torch.nn import Module, ModuleList, Sequential, Linear, Dropout, BatchNorm1d, ReLU
import torch.nn.functional as F

class HGNN(Module):
    def __init__(self, nodes_relations, parameters):
        super().__init__()
        hid           = parameters["hid"]
        layers        = parameters["layers"]
        aggregation   = parameters["aggregation"]
        dropout_p     = parameters.get("dropout", 0.1)

        # 1) stack of hetero‐message‐passing layers
        self.convs = ModuleList()
        self.bns   = ModuleList()
        self.dps   = ModuleList()
        for _ in range(layers):
            # hetero‐conv over each relation
            conv = HeteroConv(
                { rel: SAGEConv((-1, -1), aggr=aggregation, out_channels=hid, normalize=False)
                  for rel in nodes_relations },
                aggr=aggregation,
            )
            self.convs.append(conv)
            # batchnorm + dropout for the hidden dim
            self.bns.append(BatchNorm1d(hid))
            self.dps.append(Dropout(dropout_p))

        # 2) final graph‐classification head: MLP hid→hid→1
        self.classifier = Sequential(
            Linear(hid, hid),
            ReLU(),
            BatchNorm1d(hid),
            Dropout(dropout_p),
            Linear(hid, 1),
        )

    def forward(self, batch):
        x_dict    = batch.x_dict
        edge_dict = batch.edge_index_dict

        # --- message‑passing with BN/ReLU/Dropout after each conv ---
        for conv, bn, dp in zip(self.convs, self.bns, self.dps):
            x_dict = conv(x_dict, edge_dict)

            # normalize + activate + drop only on the “Activity” embeddings
            act = x_dict["Activity"]
            act = bn(act)
            act = F.relu(act)
            act = dp(act)
            x_dict["Activity"] = act

            # for all other node types, just ReLU
            for nt, x in x_dict.items():
                if nt != "Activity":
                    x_dict[nt] = F.relu(x)

        # --- graph‑level readout on “Activity” nodes ---
        h_act  = x_dict["Activity"]
        pooled = global_mean_pool(h_act, batch["Activity"].batch)

        # --- final MLP head → logits ---
        logits = self.classifier(pooled).view(-1)
        return logits



    

In [20]:
from torcheval.metrics.functional import multiclass_accuracy, multiclass_f1_score
import torch.nn as nn
import time

In [21]:
#Weighted Random Sampling
#Pull out all labels into a single 1D tensor of 0/1
y_train = torch.cat([g.y for g in X_train]).long()
#count examples per class
class_counts = torch.bincount(y_train)
#Inverse frequency
class_weights = 1.0 / class_counts.float()

print("class_counts:", class_counts.tolist())
print("class_weights:", class_weights.tolist())


# number of negatives & positives
n_neg, n_pos = class_counts.tolist()

# the weight for positive class = n_neg / n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float, device=device)
print("Using BCEWithLogitsLoss pos_weight =", pos_weight.item())

#Assign each sample the weight of it's class
sample_weights = class_weights[y_train]
#create a sampler that draws 'len(sample_weights)' samples per epoch
sampler = WeightedRandomSampler(
     weights=sample_weights,
     num_samples=len(sample_weights),
     replacement=True,
 )

class_counts: [429, 71]
class_weights: [0.0023310023825615644, 0.014084506779909134]
Using BCEWithLogitsLoss pos_weight = 6.042253494262695


In [22]:
from collections import Counter

# Draw 10,000 “indices” from the sampler
sampled_indices = list(WeightedRandomSampler(
    weights=sample_weights,
    num_samples=500,
    replacement=True
))

# Map each index back to its label
sampled_labels = [ y_train[idx].item() for idx in sampled_indices ]
print(Counter(sampled_labels))

Counter({1: 264, 0: 236})


In [23]:
from copy import deepcopy
from tqdm.notebook import tqdm

def train_hgnn(config, epochs=20):

    
    print(config)

    net = HGNN(
        parameters=config,
        nodes_relations=edge_types,
    )
    net = net.to(device)

    # loss for graph binary classification
    #loss_fn = nn.BCEWithLogitsLoss()
    loss_fn = nn.BCEWithLogitsLoss()

    #train_loader = DataLoader(X_train, batch_size=config["batch_size"], shuffle=True)
    train_loader = DataLoader(
        X_train,
        batch_size=config["batch_size"],
        sampler=sampler,     # ← use the balanced sampler
        shuffle=False,       # ← don’t shuffle when using sampler
    )


    valid_loader = DataLoader(X_valid, batch_size=config["batch_size"], shuffle=False)


    optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"])

    best_model = None
    best_loss = float("inf")
    patience = 5
    pat_count = 0

    torch.cuda.empty_cache()

    for epoch in tqdm(range(0, epochs)):
        start_time = time.time()

        #print(f"Epoch: {epoch}\n")

        net.train()
        for _, x in enumerate(train_loader):
            x = x.to(device)

            optimizer.zero_grad()       

            logits = net(x) #shape [batch_size]
            labels = x.y.float() #shape [batch_size]
            loss = loss_fn(logits, labels)

            loss.backward()
            optimizer.step()

        #--validation--
        #running_loss = 0.0
        #correct = 0
        #total = 0
        running_loss = 0.0
        all_logits = []
        all_labels = []

        net.eval()
        with torch.no_grad():
            for x in valid_loader:
                x = x.to(device)
                logits = net(x)
                labels = x.y 

                running_loss +=loss_fn(logits, labels.float()).item()

                #compute binary predictions
                #preds = (torch.sigmoid(logits) > 0.5).long()
                #correct += (preds == labels).sum().item()
                #total += labels.size(0)

                #accumulate for AUC
                all_logits.append(logits.cpu())
                all_labels.append(labels.cpu())


        val_loss = running_loss / len(valid_loader)
        #val_acc = correct / total

        #Concatenate and compute AUC
        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels).numpy()
        all_probs  = torch.sigmoid(all_logits).numpy()
        from sklearn.metrics import roc_auc_score
        val_auc = roc_auc_score(all_labels, all_probs)


        # Early stopping 
        if val_loss < best_loss:
            best_loss  = val_loss
            best_model = deepcopy(net)
            pat_count  = 0
        else:
            pat_count += 1
            if pat_count >= patience:
                break

    return best_model




In [24]:
from sklearn.metrics import roc_auc_score, f1_score
import numpy as np

def test_hgnn_multi(net):
    """
    Evaluate a trained HGNN over multiple prefix-length test sets.
    Uses global X_tests and the batch_size from net.parameters.
    Returns a dict mapping each prefix L -> AUC@L, plus the weighted-average.
    """
    net.eval()
    aucs   = []
    counts = []
    f1s = [] #+

    for L, graphs_L in X_tests.items():
        print(f"\n--- Testing prefix length L = {L} ---")
        loader = DataLoader(graphs_L, batch_size=128, shuffle=False)

        all_logits = []
        all_labels = []
        total_loss = 0.0
        loss_fn    = nn.BCEWithLogitsLoss()

        with torch.no_grad():
            for batch in loader:
                batch = batch.to(device)
                logits = net(batch)
                labels = batch.y

                total_loss += loss_fn(logits, labels.float()).item()
                all_logits.append(logits.cpu())
                all_labels.append(labels.cpu())

        # Aggregate
        avg_loss = total_loss / len(loader)
        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels).numpy()
        all_probs  = torch.sigmoid(all_logits).numpy()


        from sklearn.metrics import roc_auc_score
        auc_L = roc_auc_score(all_labels, all_probs)

        all_probs  = torch.sigmoid(all_logits).numpy().ravel()
        all_preds = (all_probs > 0.5).astype(int) #+
        f1_L = f1_score(all_labels, all_preds,zero_division=0)
        n_L   = len(graphs_L)
        print(f"AUC@{L} = {auc_L:.4f}  (n_graphs={n_L}, avg_loss={avg_loss:.4f})")

        aucs.append(auc_L)
        f1s.append(f1_L)
        counts.append(n_L)

    # Weighted-average AUC & F-score
    aucs   = np.array(aucs)
    f1s = np.array(f1s)
    counts = np.array(counts)
    weighted_auc = np.average(aucs, weights=counts)
    weighted_f1 = np.average(f1s, weights=counts)
    print(f"\n>>> Weighted-average AUC over prefixes 1–{len(aucs)}: {weighted_auc:.4f}")
    print(f">>> Weighted-average F1  over prefixes 1–{len(f1s)}: {weighted_f1:.4f}")

    return {
        **{f"AUC@{L}": a for L, a in zip(X_tests.keys(), aucs)},
        "Weighted_AUC": weighted_auc,
        "Weighted_F1": weighted_f1
    }


In [25]:
from torch_geometric.loader import DataLoader
import torch.nn as nn
import torch

def test_hgnn(net):
    """
    Evaluate a trained HGNN (graph‑level classifier) on X_test.
    Returns a dict with test loss and accuracy.
    """
    test_loader = DataLoader(X_test, batch_size=128, shuffle=False)
    loss_fn = nn.BCEWithLogitsLoss()

    net.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for x in test_loader:
            x = x.to(device)
            logits = net(x)           
            labels = x.y              

            # accumulate loss
            total_loss += loss_fn(logits, labels.float()).item()

            # binary predictions & accuracy
            #preds = (torch.sigmoid(logits) > 0.5).long()
            #correct += (preds == labels).sum().item()
            #total += labels.size(0)

            # accumulate for AUC
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())


    avg_loss = total_loss / len(test_loader)
    #accuracy = correct / total
    # compute AUC
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels).numpy()
    all_probs  = torch.sigmoid(all_logits).numpy()
    from sklearn.metrics import roc_auc_score
    test_auc = roc_auc_score(all_labels, all_probs)


    #print(f"Test loss: {avg_loss:.4f}, Test accuracy: {accuracy:.4f}")
    #return {"test_loss": avg_loss, "test_acc": accuracy}
    print(f"Test loss: {avg_loss:.4f}, Test ROC‑AUC: {test_auc:.4f}")
    return {"test_loss": avg_loss, "test_auc": test_auc}


In [26]:
# Calculate unique counts for categorical columns
list_unique = {col: len(tab_all[col].unique()) for col in categorical_columns}

#outputcat = {k : len(list_unique[k]) for k in list_unique}
outputcat = list_unique
outputreal = real_value_columns
print(outputcat)
print(outputreal)

{'Diagnose': 134, 'DiagnosticArtAstrup': 3, 'DiagnosticBlood': 3, 'DiagnosticECG': 3, 'DiagnosticIC': 3, 'DiagnosticLacticAcid': 3, 'DiagnosticLiquor': 3, 'DiagnosticOther': 3, 'DiagnosticSputum': 3, 'DiagnosticUrinaryCulture': 3, 'DiagnosticUrinarySediment': 3, 'DiagnosticXthorax': 3, 'DisfuncOrg': 3, 'Hypotensie': 3, 'Hypoxie': 3, 'InfectionSuspected': 3, 'Infusion': 3, 'Oligurie': 3, 'SIRSCritHeartRate': 3, 'SIRSCritLeucos': 3, 'SIRSCritTachypnea': 3, 'SIRSCritTemperature': 3, 'SIRSCriteria2OrMore': 3, 'CaseID': 782, 'Activity': 15, 'org:group': 25}
['Age', 'CRP', 'LacticAcid', 'Leucocytes', 'time:timestamp', 'timesincemidnight', 'month', 'weekday', 'hour', 'timesincelastevent', 'timesincecasestart', 'event_nr', 'open_cases']


In [27]:
def train_evaluate(config):
    trained_net = train_hgnn(config, epochs=50)
    #return test_hgnn    #keep for faster execution when refining parameters
    return test_hgnn_multi(trained_net)

In [28]:
import logging

logging.getLogger("root").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [29]:
y_train = torch.cat([batch.y for batch in X_train]).float()
num_true = y_train.sum().item()
num_false = len(y_train) - num_true

# Assign weights to BCEWithLogitsLoss
pos_weight = num_false / num_true
pos_weight = torch.tensor([num_false / num_true], device=device)

print("pos_weight: ", pos_weight)

pos_weight:  tensor([6.0423])


In [30]:
sample_config = {
    "hid":          128, #128
    "layers":       2, #2
    "lr":           1e-3,
    "batch_size":   256, #256
    "aggregation": "mean",
}


# 1-epoch train just to exercise the code-path and see prints
net = train_hgnn(sample_config, epochs=1)

# run the test (with your debug prints enabled)
res = test_hgnn(net)
print("test_hgnn returned:", res)
print("\n")

# run the test, multi version
res = test_hgnn_multi(net)
print("test_hgnn returned:", res)


{'hid': 128, 'layers': 2, 'lr': 0.001, 'batch_size': 256, 'aggregation': 'mean'}


  0%|          | 0/1 [00:00<?, ?it/s]

Test loss: 2.6508, Test ROC‑AUC: 0.4968
test_hgnn returned: {'test_loss': 2.650842010974884, 'test_auc': 0.4968013468013468}



--- Testing prefix length L = 1 ---
AUC@1 = 0.5721  (n_graphs=157, avg_loss=2.8460)

--- Testing prefix length L = 2 ---
AUC@2 = 0.5000  (n_graphs=157, avg_loss=18.9085)

--- Testing prefix length L = 3 ---
AUC@3 = 0.5000  (n_graphs=157, avg_loss=18.9086)

--- Testing prefix length L = 4 ---
AUC@4 = 0.5000  (n_graphs=157, avg_loss=18.9086)

--- Testing prefix length L = 5 ---
AUC@5 = 0.5000  (n_graphs=156, avg_loss=19.2625)

--- Testing prefix length L = 6 ---
AUC@6 = 0.5000  (n_graphs=155, avg_loss=19.3328)

--- Testing prefix length L = 7 ---
AUC@7 = 0.5000  (n_graphs=154, avg_loss=19.4023)

--- Testing prefix length L = 8 ---
AUC@8 = 0.5000  (n_graphs=154, avg_loss=19.4025)

--- Testing prefix length L = 9 ---
AUC@9 = 0.5000  (n_graphs=143, avg_loss=20.5568)

--- Testing prefix length L = 10 ---
AUC@10 = 0.5000  (n_graphs=135, avg_loss=20.8970)

--- Testing

In [31]:
import warnings
warnings.filterwarnings("ignore", message="Untracked metric.*")

best_parameters, values, experiment, model = optimize(
    parameters=[
        #{"name": "hid", "type": "choice", "values": [64,128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False},
        {"name": "hid", "type": "choice", "values": [128], "value_type": "int", "is_ordered" : True,"sort_values":False},
        {"name": "layers", "type": "choice", "values": [2, 3], "value_type": "int", "is_ordered" : True, "sort_values":False},
        #{"name": "layers", "type": "choice", "values": [2], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "lr", "type": "range", "bounds": [1e-4, 1e-1], "value_type": "float", "log_scale": True},
        #{"name": "batch_size", "type": "choice", "values": [128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False}, 
        {"name": "batch_size", "type": "choice", "values": [16,32,64], "value_type": "int", "is_ordered" : True,"sort_values":False}, 
        #{"name": "heads", "type": "choice", "values": [1,2], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "heads", "type": "choice", "values": [1], "value_type": "int", "is_ordered" : True,"sort_values":False},
        
        {"name": "aggregation", "type" : "choice", "values" :["sum", "mean", "max"], "value_type" : "str"}
        #{"name": "aggregation", "type" : "choice", "values" :["max"], "value_type" : "str"},
     
    ],
  
    evaluation_function=train_evaluate,
    objective_name='Weighted_AUC', #test_auc for single/multi switch
    arms_per_trial=1,
    minimize = False,
    random_seed = 123,
    total_trials = 30
)

print(best_parameters)
means, covariances = values
print(means)
print(experiment)

[INFO 07-17 19:16:10] ax.service.utils.instantiation: Choice parameter hid contains only one value, converting to a fixed parameter instead.
/home/matteo/Documents/GNN-test2/SEPH_MODELS/env3/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `is_ordered` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False`  since the parameter is a string with more than 2 choices.. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction. Note that choice parameters with exactly 2 choices are always considered ordered and that the user-supplied `is_ordered` has no effect in this particular case.
  return ChoiceParameter(
/home/matteo/Documents/GNN-test2/SEPH_MODELS/env3/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False` for parameters of `ParameterType` STRING. To ov

{'layers': 2, 'lr': 0.005537727881833665, 'batch_size': 32, 'aggregation': 'sum', 'hid': 128}


  0%|          | 0/50 [00:00<?, ?it/s]


--- Testing prefix length L = 1 ---
AUC@1 = 0.4281  (n_graphs=157, avg_loss=5.7722)

--- Testing prefix length L = 2 ---
AUC@2 = 0.5717  (n_graphs=157, avg_loss=1.0861)

--- Testing prefix length L = 3 ---
AUC@3 = 0.5717  (n_graphs=157, avg_loss=0.4434)

--- Testing prefix length L = 4 ---
AUC@4 = 0.5717  (n_graphs=157, avg_loss=0.3532)

--- Testing prefix length L = 5 ---
AUC@5 = 0.5922  (n_graphs=156, avg_loss=0.3201)

--- Testing prefix length L = 6 ---
AUC@6 = 0.6181  (n_graphs=155, avg_loss=0.3228)

--- Testing prefix length L = 7 ---
AUC@7 = 0.6090  (n_graphs=154, avg_loss=0.3222)

--- Testing prefix length L = 8 ---
AUC@8 = 0.6090  (n_graphs=154, avg_loss=0.3338)

--- Testing prefix length L = 9 ---
AUC@9 = 0.5622  (n_graphs=143, avg_loss=0.1529)

--- Testing prefix length L = 10 ---
AUC@10 = 0.6123  (n_graphs=135, avg_loss=0.0967)

--- Testing prefix length L = 11 ---
AUC@11 = 0.7658  (n_graphs=124, avg_loss=0.1133)

--- Testing prefix length L = 12 ---
AUC@12 = 0.7667  (n_gra

[INFO 07-17 19:17:19] ax.core.experiment: Attached data has some metrics ({'AUC@11', 'AUC@6', 'AUC@12', 'AUC@5', 'AUC@9', 'AUC@10', 'AUC@14', 'AUC@1', 'AUC@2', 'AUC@4', 'AUC@8', 'AUC@3', 'AUC@13', 'AUC@7', 'AUC@16', 'AUC@15', 'Weighted_F1'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[INFO 07-17 19:17:19] ax.service.managed_loop: Running optimization trial 2...
[ERROR 07-17 19:17:19] ax.core.observation: Data contains metric AUC@1 that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric AUC@1.
NoneType: None
[ERROR 07-17 19:17:19] ax.core.observation: Data con

...


--- Testing prefix length L = 1 ---
AUC@1 = 0.5000  (n_graphs=157, avg_loss=2238746.6250)

--- Testing prefix length L = 2 ---
AUC@2 = 0.3926  (n_graphs=157, avg_loss=1.7741)

--- Testing prefix length L = 3 ---
AUC@3 = 0.4007  (n_graphs=157, avg_loss=1.7732)

--- Testing prefix length L = 4 ---
AUC@4 = 0.4013  (n_graphs=157, avg_loss=1.7728)

--- Testing prefix length L = 5 ---
AUC@5 = 0.4370  (n_graphs=156, avg_loss=1.5104)

--- Testing prefix length L = 6 ---
AUC@6 = 0.4656  (n_graphs=155, avg_loss=1.4567)

--- Testing prefix length L = 7 ---
AUC@7 = 0.4752  (n_graphs=154, avg_loss=1.4065)

--- Testing prefix length L = 8 ---
AUC@8 = 0.4702  (n_graphs=154, avg_loss=1.4080)

--- Testing prefix length L = 9 ---
AUC@9 = 0.4345  (n_graphs=143, avg_loss=0.7266)

--- Testing prefix length L = 10 ---
AUC@10 = 0.5585  (n_graphs=135, avg_loss=0.3163)

--- Testing prefix length L = 11 ---
AUC@11 = 0.6364  (n_graphs=124, avg_loss=0.3984)

--- Testing prefix length L = 12 ---
AUC@12 = 0.4697  

[INFO 07-17 19:47:54] ax.core.experiment: Attached data has some metrics ({'AUC@11', 'AUC@6', 'AUC@12', 'AUC@5', 'AUC@9', 'AUC@10', 'AUC@14', 'AUC@1', 'AUC@2', 'AUC@4', 'AUC@8', 'AUC@3', 'AUC@13', 'AUC@7', 'AUC@16', 'AUC@15', 'Weighted_F1'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[ERROR 07-17 19:47:54] ax.core.observation: Data contains metric AUC@1 that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric AUC@1.
NoneType: None
[ERROR 07-17 19:47:54] ax.core.observation: Data contains metric AUC@10 that has not been added to the experiment. You can either u

AUC@16 = 0.6124  (n_graphs=46, avg_loss=1.3757)

>>> Weighted-average AUC over prefixes 1–16: 0.4816
>>> Weighted-average F1  over prefixes 1–16: 0.0189


[ERROR 07-17 19:47:55] ax.core.observation: Data contains metric AUC@5 that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric AUC@5.
NoneType: None
[ERROR 07-17 19:47:55] ax.core.observation: Data contains metric AUC@6 that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric AUC@6.
NoneType: None
[ERROR 07-17 19:47:55] ax.core.observation: Data contains metric AUC@7 that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric AUC@7.
NoneType: None
[ERROR 07-17 19:47:55] ax.core.

{'layers': 2, 'lr': 0.00014041306205462534, 'batch_size': 64, 'aggregation': 'max', 'hid': 128}
{'AUC@1': 0.5, 'AUC@2': 0.5208754208754209, 'AUC@3': 0.5777777777777777, 'AUC@4': 0.6828282828282829, 'AUC@5': 0.6955908289241622, 'AUC@6': 0.7033333333333334, 'AUC@7': 0.6771929824561403, 'AUC@8': 0.6982456140350877, 'AUC@9': 0.7844112769485904, 'AUC@10': 0.8061538461538462, 'AUC@11': 0.650137741046832, 'AUC@12': 0.6363636363636364, 'AUC@13': 0.6418439716312057, 'AUC@14': 0.6396396396396397, 'AUC@15': 0.7062146892655367, 'AUC@16': 0.7131782945736433, 'Weighted_AUC': 0.660063285151528, 'Weighted_F1': 0.006679145750021272}
Experiment(None)


In [32]:
from ax.service.utils.report_utils import exp_to_df

results = exp_to_df(experiment)
#results.sort_values(by="test_auc")
#results = results.sort_values(by="test_auc")
results.sort_values(by="Weighted_AUC")
results = results.sort_values(by="Weighted_AUC")
results.to_csv(f"results/{dataset}.csv", sep=",")

[WARNING 07-17 19:47:56] ax.service.utils.report_utils: Column reason missing for all trials. Not appending column.
